In [6]:
import os
import sys
from pathlib import Path
import json
from typing import Dict

import numpy as np
import pandas as pd

# Detect the workspace root and add src/ to sys.path.
notebook_dir = Path.cwd()
if notebook_dir.name == "dataset":
    workspace_root = notebook_dir.parent
elif (notebook_dir / "src").exists():
    workspace_root = notebook_dir
elif (notebook_dir.parent / "src").exists():
    workspace_root = notebook_dir.parent
else:
    raise FileNotFoundError("Unable to locate the PLACy workspace root containing src/")

src_root = workspace_root / "src"
sys.path.insert(0, str(src_root))

from data.ou_process_stocklike import ou_process_stocklike
from data.ou_process import ou_process


for sb in [0.0, 0.1, 0.5, 1.0]:
    for stationary in [True, False]:
        for stock_like in [True, False]:
            for seed in [i for i in range(0,10)]:

                params = dict(
                    seed=seed,
                    n_vars=5,
                    length=5000,
                    prob_edge=0.5,
                    lag=5,
                    causal_strength=0.5,
                    s_g=1.0,
                    s_b=sb,
                    tau_c=5.0,
                    noise_exponent=2.0,
                    decay_exponent=1.0,
                    causal_exponent=1,
                    post_causality=True,
                    stationary=stationary,
                    stock_like=stock_like,
                    dir_name="Benchmark",
                    data_dir="dataset"
                )

                np.random.seed(params["seed"])

                if params["stock_like"]:
                    d: Dict[str, np.ndarray] = ou_process_stocklike(
                        n_vars=params["n_vars"],
                        length=params["length"],
                        prob_edge=params["prob_edge"],
                        lag=params["lag"],
                        causal_strength=params["causal_strength"],
                        post_causality=params["post_causality"],
                        σ_g=params["s_g"],
                        σ_b=params["s_b"],
                        tau_c=params["tau_c"],
                        noise_exponent=params["noise_exponent"],
                        decay_exponent=params["decay_exponent"],
                        causal_exponent=params["causal_exponent"],
                        stationary=params["stationary"],
                        plot=False,
                    )
                else:
                    d: Dict[str, np.ndarray] = ou_process(
                        n_vars=params["n_vars"],
                        length=params["length"],
                        prob_edge=params["prob_edge"],
                        lag=params["lag"],
                        causal_strength=params["causal_strength"],
                        post_causality=params["post_causality"],
                        σ_g=params["s_g"],
                        σ_b=params["s_b"],
                        tau_c=params["tau_c"],
                        noise_exponent=params["noise_exponent"],
                        decay_exponent=params["decay_exponent"],
                        causal_exponent=params["causal_exponent"],
                        stationary=params["stationary"],
                        plot=False,
                    )

                process = d["process"]
                causal_matrix = d["causal_matrix"]

                # Build the save path using the instruction inside the notebook
                base_path = workspace_root / params["data_dir"]

                data_type = "stat" if params["stationary"] else "nonstat"
                if params["stock_like"]:
                    data_type += "_stocklike"

                path = (
                    base_path
                    / params["dir_name"]
                    / f"{data_type}"
                    / f"n_vars={params['n_vars']}"
                    / f"causal_strength={params['causal_strength']}"
                    / f"s_g={params['s_g']}"
                    / f"s_b={params['s_b']}"
                    / f"seed={params['seed']}"
                )
                path.mkdir(parents=True, exist_ok=True)

                # Save the multivariate timeseries in the same orientation as example.csv:
                # rows = timestamps, columns = variables, with headers X0, X1, ...
                process_df = pd.DataFrame(
                    process.T,
                    columns=[f"X{i}" for i in range(process.shape[0])]
                )
                process_df.to_csv(path / "process.csv", index=False)

                # Save the causal adjacency matrix as both .npy and .csv
                np.save(path / "adj_matrix.npy", causal_matrix)
                causal_df = pd.DataFrame(
                    causal_matrix,
                    columns=[f"X{i}" for i in range(causal_matrix.shape[1])]
                )
                causal_df.to_csv(path / "adj_matrix.csv", index=False)

                metadata = {
                    "params": params,
                    "process_shape": process_df.shape,
                    "causal_matrix_shape": causal_matrix.shape,
                }
                with open(path / "metadata.json", "w") as f:
                    json.dump(metadata, f, indent=2)